# Positions from the delays, by high voltage

*Third notebook of `relcal`. Draft prepared with Claude Code for A. Villano, 2026-10-30.*

The series of Run 76 were taken at different high voltages (0 to -73 V). `positions-from-delays.ipynb` placed the events of one series (-73 V) on the face with the delays `xdel`, `ydel` between the 20 % rise times of the channels. Here the processed series are grouped by voltage (`series.csv` of the output folder), and the density of events in the plane of the delays is drawn for every group, with the detector underneath, so the groups can be compared.

The definitions are those of `positions-from-delays.ipynb`: `xdel = t_E - (t_C + t_D) / 2 - 1 us`, `ydel = 0.866 (t_D - t_C) + 9 us`, with the empirical offsets of Fritts's code, and the times from the raw traces (`rise_crossing`, `fraction = 0.2`), not filtered. Real triggers only (`Unknown`; the label `Physics` marks the random triggers, see `integral-on-the-face.ipynb`), taken from the `trigger_type` column that `goldsluice run` now writes.

**Needs** the output of `data_analysis/R76/processing/r76-d-idx.dataset.toml` **with a `rise_crossing` step** (`fraction = 0.2`) for every series. Without it the notebook stops at the second cell and says so. Run with the `darkmatter_cli_env` kernel.

In [ ]:
import csv
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colors
from matplotlib.patches import Wedge

from goldpan.dataset import Dataset

In [ ]:
_here = Path.cwd().resolve()
DEFINITION = next(p / "data_analysis" / "R76" / "processing" / "r76-d-idx.dataset.toml"
                  for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
ds = Dataset.open(DEFINITION)
OUT = Path(ds.spec.output)
LETTERS = "ABCDEF"

q = ds.processed()                                        # the quantities only, no pulses
if "crossing_time_s" not in q.quantities["Side2pA"]:
    raise SystemExit(f"{OUT} has no crossing_time_s: add the step `rise_crossing` (fraction = 0.2) to {DEFINITION.name} "
                     f"and run `python -m goldsluice run` on it (it redoes only that algorithm)")
series = np.asarray(q.quantities["series"]).astype(str)
trigger = np.asarray(q.quantities["trigger_type"]).astype(str)
t = {k: np.asarray(q.quantities[f"Side2p{k}"]["crossing_time_s"], dtype=float) * 1e6 for k in LETTERS}     # microseconds
integral = np.column_stack([np.asarray(q.quantities[f"Side2p{k}"]["integral"], dtype=float) for k in LETTERS])
print(f"{len(series):,} events in {len(np.unique(series))} series; trigger types: "
      f"{ {str(k): int((trigger == k).sum()) for k in np.unique(trigger)} }")

## The voltage groups

`series.csv` lists the high voltage of every series (from the NSDF series table). Series whose voltages differ by less than `TOL` volts are one group. The trigger setting (`DCRC1pA 2` or `100`) varies within some groups; it is listed so it can be told apart.

In [ ]:
TOL = 1.0                                  # volts: series closer than this are one group

with open(OUT / "series.csv", newline="") as fh:
    meta = {r["series"]: r for r in csv.DictReader(fh)}
hv_of = {s: float(meta[s]["hv_v"]) for s in np.unique(series)}
levels = []                                # the group voltages, from 0 downward
for v in sorted(set(hv_of.values()), reverse=True):
    if not levels or abs(levels[-1] - v) >= TOL:
        levels.append(v)
group_of = {s: min(levels, key=lambda g: abs(g - hv_of[s])) for s in hv_of}
hv = np.array([group_of[s] for s in series])
print(f"{'group (V)':>10} {'series':>7} {'events':>11}  triggers")
for g in levels:
    ss = [s for s in hv_of if group_of[s] == g]
    print(f"{g:>10.1f} {len(ss):>7} {(hv == g).sum():>11,}  " + ", ".join(sorted({meta[s]['trigger'] for s in ss})))

## Delays and selection

As in `positions-from-delays.ipynb`: real triggers, the sum of the positive offset-subtracted integrals of the six channels above `THRESH`, six times that exist, and all six within `WINDOW` of their median (one pulse). The offset of an integral is its median **within the series**, not over the whole run, since the baseline may differ from series to series.

In [ ]:
S = integral.copy()
for s in np.unique(series):
    m = series == s
    S[m] -= np.median(integral[m], axis=0)
total = np.clip(S, 0, None).sum(axis=1)

THRESH = 2.0                       # as in integral-on-the-face.ipynb
WINDOW = 60.0                      # microseconds
OFFSET_X, OFFSET_Y = -1.0, 9.0     # Fritts's empirical offsets

T = np.column_stack([t[k] for k in LETTERS])
xdel = t["E"] - 0.5 * (t["C"] + t["D"]) + OFFSET_X
ydel = 0.866 * (t["D"] - t["C"]) + OFFSET_Y
phidel = np.degrees(np.arctan2(ydel, xdel))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    one_pulse = (np.abs(T - np.nanmedian(T, axis=1, keepdims=True)) < WINDOW).all(axis=1)
use = (trigger == "Unknown") & (total > THRESH) & np.isfinite(T).all(axis=1) & one_pulse
print(f"{'group (V)':>10} {'real trig.':>11} {'used':>9} {'fraction':>9} {'x median':>9} {'x width':>8} {'y median':>9} {'y width':>8}  (widths: 16-84 % half range, us)")
for g in levels:
    a, m = (hv == g), (hv == g) & use
    pc = lambda v: np.percentile(v[m], [16, 50, 84]) if m.any() else np.full(3, np.nan)
    px, py = pc(xdel), pc(ydel)
    print(f"{g:>10.1f} {(a & (trigger == 'Unknown')).sum():>11,} {m.sum():>9,} {m.sum() / max(a.sum(), 1):>9.1%} "
          f"{px[1]:>9.2f} {(px[2] - px[0]) / 2:>8.2f} {py[1]:>9.2f} {(py[2] - py[0]) / 2:>8.2f}")

## The detector in the plane of the delays

Drawn as in `positions-from-delays.ipynb`: the `x` axis is reversed so the face has the orientation of the first notebook (`C` upper left, `D` lower left, `E` right). The scale is a placeholder, `R_US` microseconds for the radius of the face.

In [ ]:
REGIONS = {   # letter -> (inner radius, outer radius, start angle, end angle) in degrees, radius 1 = the face
    "A": (0.90, 1.00, 0, 360), "B": (0.80, 0.90, 0, 360),
    "C": (0.25, 0.80, 60, 180), "D": (0.25, 0.80, 180, 300), "E": (0.25, 0.80, -60, 60),
    "F": (0.00, 0.25, 0, 360),
}
R_US = 14.0       # PLACEHOLDER: the radius of the face in microseconds of delay

def draw_detector_mirrored(ax, radius=R_US, label=True, lw=0.8):
    """The face of radius `radius` centred on the origin, mirrored in x, for an axis whose x is reversed."""
    for k, (r_in, r_out, a0, a1) in REGIONS.items():
        ax.add_patch(Wedge((0, 0), r_out * radius, 180 - a1, 180 - a0, width=(r_out - r_in) * radius,
                           facecolor="none", edgecolor="k", linewidth=lw, zorder=3))
        if label and k in "CDE":
            a = np.deg2rad((a0 + a1) / 2)
            ax.text(-0.52 * radius * np.cos(a), 0.52 * radius * np.sin(a), k, ha="center", va="center", fontsize=11, color="0.25", zorder=5)
    ax.set_aspect("equal")

XR, YR = 22, 16
n = len(levels)
cols = 3
rows = -(-n // cols)
fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 4.2 * rows), squeeze=False)
for ax, g in zip(axes.flat, levels):
    m = use & (hv == g)
    h = ax.hist2d(xdel[m], ydel[m], bins=[110, 80], range=[[-XR, XR], [-YR, YR]], cmap="viridis", norm=colors.LogNorm(), cmin=1, zorder=2)
    draw_detector_mirrored(ax)
    ax.set_xlim(XR, -XR)
    ax.set_ylim(-YR, YR)
    ax.set_title(f"{g:.1f} V: {m.sum():,} events", fontsize=11)
    ax.set_xlabel("xdel (us, reversed)")
    ax.set_ylabel("ydel (us)")
    fig.colorbar(h[3], ax=ax, shrink=0.8)
for ax in list(axes.flat)[n:]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## The same, each group normalised

The number of events differs by a large factor between groups, so the colour scale above is not comparable. Here every panel is the fraction of the group's events per bin on one common scale.

In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 4.2 * rows), squeeze=False)
dens = []
for g in levels:
    m = use & (hv == g)
    d, xe, ye = np.histogram2d(xdel[m], ydel[m], bins=[110, 80], range=[[-XR, XR], [-YR, YR]])
    dens.append(d / max(m.sum(), 1))
vmax = max(d.max() for d in dens)
for ax, g, d in zip(axes.flat, levels, dens):
    c = ax.pcolormesh(xe, ye, np.where(d > 0, d, np.nan).T, cmap="magma", norm=colors.LogNorm(vmin=vmax * 1e-3, vmax=vmax), zorder=2)
    draw_detector_mirrored(ax)
    ax.set_xlim(XR, -XR)
    ax.set_ylim(-YR, YR)
    ax.set_title(f"{g:.1f} V", fontsize=11)
    fig.colorbar(c, ax=ax, shrink=0.8, label="fraction of the group's events per bin")
for ax in list(axes.flat)[n:]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## Azimuth

`phidel = atan2(ydel, xdel)` for each group, as histograms of the fraction of the events per bin.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
cmap = plt.get_cmap("viridis")
for i, g in enumerate(levels):
    m = use & (hv == g)
    ax.hist(phidel[m], bins=np.linspace(-180, 180, 91), histtype="step", density=True, color=cmap(i / max(n - 1, 1)), label=f"{g:.1f} V")
ax.set_xlabel("phidel (degrees)")
ax.set_ylabel("fraction of events per degree")
ax.legend(ncol=3, fontsize=8)
plt.show()

**What this shows, and what is not done.** The density of events in the plane of the delays at each voltage, with the detector's regions drawn at a placeholder scale. Not done: millimetres (`R_US` is a placeholder), the offsets (taken from Fritts's code), filtered times, and a check that the readout and the pulse position are the same for all series (see the README). The trigger setting changes within some groups; if a pattern follows the trigger and not the voltage, split the group by `meta[s]["trigger"]`.